# Practica 4 - Pruebas estadisticas

In [ ]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import statsmodels.api as sm
from statsmodels.formula.api import ols
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import combinations

path = "../data/procesed/metro.csv"
metro = pd.read_csv(path)
metro.head()

## Seleccion de datos

In [ ]:
top_categories = ["Clouds", "Clear", "Rain", "Mist", "Snow"]
df_filtered = metro[metro["weather_main"].isin(top_categories)].copy()

df_sample = df_filtered.sample(n=2000, random_state=42).reset_index(drop=True)
df_sample["weather_main"].value_counts()

Para aplicar ANOVA se revisan estos supuestos:

1. Normalidad de los residuos.
2. Homocedasticidad entre grupos.

Hipotesis de ANOVA:

- H0: las medias del volumen de trafico son iguales entre las categorias climaticas.
- H1: al menos una categoria tiene una media diferente.

In [ ]:
modelo_ols = ols("traffic_volume ~ C(weather_main)", data=df_sample).fit()

residuos = modelo_ols.resid
stat_shapiro, p_shapiro = stats.shapiro(residuos)

grupos = [grupo["traffic_volume"].values for _, grupo in df_sample.groupby("weather_main")]
stat_levene, p_levene = stats.levene(*grupos)

print("verificacion de supuestos:")
print(f"1. Normalidad: Estadistico = {stat_shapiro:.4f}, p-valor = {p_shapiro:.4e}")
print(f"2. Homocedasticida:  Estadistico = {stat_levene:.4f}, p-valor = {p_levene:.4f}")

cumple_normalidad = p_shapiro > 0.05
cumple_homocedasticidad = p_levene > 0.05

print("")
print("Resultado de evaluacion:")
print(f"cumple supuesto de Normalidad (p > 0.05)?: {cumple_normalidad}")
print(f"cumple supuesto de Homocedasticidad (p > 0.05)?: {cumple_homocedasticidad}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

sns.histplot(residuos, kde=True, ax=axes[0], color="skyblue")
axes[0].set_title("Distribucion de residuos del modelo")
axes[0].set_xlabel("residuos")

stats.probplot(residuos, dist="norm", plot=axes[1])
axes[1].set_title("Grafico Q-Q de residuos")

plt.tight_layout()
plt.show()

## Tabla ANOVA

Los residuos no cumplen normalidad y las varianzas no son iguales. Por eso la tabla ANOVA solo se usa como referencia y se continua con Kruskal-Wallis.

In [ ]:
tabla_anova = sm.stats.anova_lm(modelo_ols, typ=2)
print("=== TABLA ANOVA ===")
tabla_anova

## Prueba de Kruskal-Wallis

Se compara la distribucion del volumen de trafico entre las categorias de clima.

- H0: la distribucion es igual en todas las categorias.
- H1: al menos una categoria tiene una distribucion diferente.

In [ ]:
stat_kw, p_kw = stats.kruskal(*grupos)

print("=== PRUEBA DE KRUSKAL-WALLIS ===")
print(f"Estadistico H de Kruskal-Wallis: {stat_kw:.4f}")
print(f"p-valor: {p_kw:.4e}")

alpha = 0.05
if p_kw < alpha:
    print(f"Conclusion (p < {alpha}): Se rechaza la hipotesis nula H0.")
    print("Existen diferencias estadisticamente significativas en el volumen de trafico entre los tipos de clima.")
else:
    print(f"Conclusion (p >= {alpha}): No se rechaza la hipotesis nula H0.")
    print("no hay evidencia suficiente de diferencias significativas entre los tipos de clima.")

## Comparaciones por pares

Si Kruskal-Wallis es significativa, se aplican pruebas U de Mann-Whitney por pares con correccion de Bonferroni.

In [ ]:
categorias_unicas = df_sample["weather_main"].unique()
parejas = list(combinations(categorias_unicas, 2))
m = len(parejas)
alpha = 0.05
alpha_bonferroni = alpha / m

print(f"Numero de comparaciones pareadas: {m}")
print(f"Nivel de significancia ajustado (Bonferroni): alpha_adj = {alpha_bonferroni:.4f}")

resultados_posthoc = []
for g1, g2 in parejas:
    datos_g1 = df_sample[df_sample["weather_main"] == g1]["traffic_volume"]
    datos_g2 = df_sample[df_sample["weather_main"] == g2]["traffic_volume"]
    
    u_stat, p_val = stats.mannwhitneyu(datos_g1, datos_g2, alternative="two-sided")
    es_significativo = p_val < alpha_bonferroni
    
    resultados_posthoc.append({
        "Grupo 1": g1,
        "grupo 2": g2,
        "estadistico U": u_stat,
        "p-valor": p_val,
        "Diferencia Significativa": es_significativo
    })

df_posthoc = pd.DataFrame(resultados_posthoc)
df_posthoc

## Dia laboral y fin de semana

Se compara el volumen de trafico entre dias laborales y fines de semana. La prueba se elige segun los supuestos.

In [ ]:
metro["date_time_dt"] = pd.to_datetime(metro["date_time"])
metro["tipo_dia"] = metro["date_time_dt"].dt.dayofweek.isin([5, 6]).map({True: "Fin de semana", False: "Dia laboral"})

sample_2g = metro.sample(n=1000, random_state=42)
g_laboral = sample_2g[sample_2g["tipo_dia"] == "Dia laboral"]["traffic_volume"]
g_finde = sample_2g[sample_2g["tipo_dia"] == "Fin de semana"]["traffic_volume"]

_, p_norm_lab = stats.shapiro(g_laboral)
_, p_norm_finde = stats.shapiro(g_finde)

_, p_lev_2g = stats.levene(g_laboral, g_finde)

print("=== VERIFICACION DE REQUISITOS (2 GRUPOS) ===")
print(f"Normalidad Dia laboral (p-valor): {p_norm_lab:.4e}")
print(f"Normalidad Fin de semana (p-valor): {p_norm_finde:.4e}")
print(f"Homocedasticidad Levene (p-valor): {p_lev_2g:.4e}")

if p_norm_lab > 0.05 and p_norm_finde > 0.05 and p_lev_2g > 0.05:
    t_stat, p_t = stats.ttest_ind(g_laboral, g_finde)
    print(f"-> Se utilizo PRUEBA T DE STUDENT: t = {t_stat:.4f}, p-valor = {p_t:.4e}")
else:
    u_stat_2g, p_u_2g = stats.mannwhitneyu(g_laboral, g_finde, alternative="two-sided")
    print(f"-> Los datos NO cumplen normalidad/homocedasticidad. Se utilizo PRUEBA U DE MANN-WHITNEY:")
    print(f"   Estadistico U = {u_stat_2g:.4f}, p-valor = {p_u_2g:.4e}")

## Graficas por grupo

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=df_sample, x="weather_main", y="traffic_volume", hue="weather_main", ax=axes[0], palette="Set2", legend=False)
axes[0].set_title("volumen de Trafico por Categoria Climatica")
axes[0].set_xlabel("categoria de Clima")
axes[0].set_ylabel("Volumen de Trafico")

sns.violinplot(data=sample_2g, x="tipo_dia", y="traffic_volume", hue="tipo_dia", ax=axes[1], palette="coolwarm", legend=False)
axes[1].set_title("Volumen de Trafico: Dia Laboral vs Fin de Semana")
axes[1].set_xlabel("Tipo de Dia")
axes[1].set_ylabel("Volumen de Trafico")

plt.tight_layout()
plt.show()

## Conclusiones

- Los supuestos de normalidad y homocedasticidad no se cumplen.
- Kruskal-Wallis permite comparar los grupos sin esos supuestos.
- La comparacion entre dia laboral y fin de semana usa Mann-Whitney cuando no se cumplen los requisitos parametricos.